# Prep route and stop gdf
1. only subset for operators by region
2. do aggregation?
3. subset to special routes
   if this is skipped, after step 2, can go straight to regression?

In [1]:
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2

credentials, _ = google.auth.default()

In [3]:
route_gdf = D1.prep_fct_daily_schedule_rt_route_direction_summary(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    event_time_of_day_dict = wc_vars.sofi_match_times,
) 

In [5]:
route_gdf.schedule_name.value_counts()

schedule_name
LA Metro Bus Schedule        8624
LA DOT Schedule              3170
Big Blue Bus Schedule        1173
Culver City Schedule          534
Torrance Schedule             532
Metrolink Schedule            525
LA Metro Rail Schedule        468
G Trans Schedule              380
Beach Cities GMV Schedule     156
LA Metro Events Schedule       30
Name: count, dtype: int64

In [8]:
sofi_trips = D1.filter_to_special_routes(
    route_gdf,
    route_name_dict = wc_vars.special_socal_routes_dict
)

In [9]:
trips_by_event = D2.aggregate_by_event_type(
    sofi_trips, 
    group_cols = ["schedule_name", "route_name", "direction_id", "event_day", "day_type"], 
    metric_cols = ["n_trips"]
).rename(columns = {
    "n_trips": "daily_trips", 
})

trips_wide = D2.make_wide(
    trips_by_event,
    index_cols=["schedule_name", "route_name", "direction_id"],
    pivot_cols=["day_type", "event_day"],
    value_cols=["daily_trips"],
)

In [13]:
trips_by_event2 = D2.aggregate_by_event_type(
    route_gdf, 
    group_cols = ["schedule_name", "route_name", "direction_id", "event_day", "day_type"], 
    metric_cols = ["n_trips"]
).rename(columns = {
    "n_trips": "daily_trips", 
})

trips_wide2 = D2.make_wide(
    trips_by_event,
    index_cols=["schedule_name", "route_name", "direction_id"],
    pivot_cols=["day_type", "event_day"],
    value_cols=["daily_trips"],
)

In [17]:
# on the full df, this makes sense, most of them are 0s
trips_wide2.change_daily_trips_weekday.value_counts(normalize=True)

change_daily_trips_weekday
 0.0      0.692632
-0.1      0.037895
 0.2      0.014737
 0.3      0.012632
-0.3      0.010526
            ...   
-2.5      0.002105
 25.0     0.002105
 35.0     0.002105
 174.0    0.002105
 0.7      0.002105
Name: proportion, Length: 77, dtype: float64